# Phase 1 — Train text-aware DTD (loss + OCR prior)

Fine-tunes the DocTamper **DTD** model with our **`CombinedTamperLoss`** (CE + Dice + boundary), and
documents how to inject the **OCR text prior** (`TextPriorFusion`) into the decoder.

## ⚠️ Read before running
- This **runs the real DocTamper model**, so it `torch.load`s the checkpoint + ImageNet backbones and
  `pickle.load`s `qt_table.pk` / `pks/*.pk`. These are *untrusted* files — run **only in this
  disposable Colab session**, never on your laptop, never with secrets in the mounted Drive.
- DocTamper's pinned stack (torch 1.11) **does not install on current Colab Python**, so we run on
  Colab's **modern torch** and install only the libraries the code needs (the setup cell handles the
  known conflicts: the efficientnet/smp clash and the wrong-`jpegio` issue).
- You must download the author's **checkpoints** folder to `MyDrive/checkpoints` (see the stage cell):
  `vph_imagenet.pt` + `swin_imagenet.pt` are loaded when the model is *built*; `dtd_doctamper.pth` is
  for fine-tuning.
- The forward call mirrors `eval_dtd.py` exactly: `model(image, dct_int32, q.unsqueeze(1))`.

In [1]:
# --- Setup: clone repos + install the stack (modern torch is fine) ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys
%cd /content
if not os.path.exists('/content/DocTamper'):
    !git clone -q https://github.com/qcf-568/DocTamper.git
if not os.path.exists('/content/HLCV-Project'):
    !git clone -q https://github.com/SamiraAbedini/HLCV-Project.git

# Libs the code needs. Do NOT pin torch (use Colab's).
!pip -q install lmdb six albumentations timm==0.4.12 segmentation_models_pytorch==0.2.1 easyocr
# DTD needs efficientnet 0.7.1's API. smp 0.2.1 pins 0.6.3 (which breaks Conv2dStaticSamePadding),
# so install 0.7.1 SEPARATELY to override it -- a one-command install would fail to resolve.
# pip will warn that smp is unhappy; that is expected and matches the author's working setup.
!pip -q install efficientnet_pytorch==0.7.1

# Correct jpegio = dwgoon's (provides jpegio.read used by DocTamper). The PyPI 'jpegio'
# lacks .read, so build from source. Installed BEFORE anything imports jpegio (no restart).
!pip -q uninstall -y jpegio
!apt-get -qq install -y libjpeg-dev > /dev/null
!rm -rf /content/jpegio && git clone -q https://github.com/dwgoon/jpegio.git /content/jpegio
%cd /content/jpegio
!pip -q install .
%cd /content/DocTamper/models

sys.path.append('/content/HLCV-Project')   # our src/ modules
import jpegio
print('jpegio.read available:', hasattr(jpegio, 'read'))
print('setup done. If jpegio.read is False, do Runtime > Restart and re-run this cell once.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
segmentation-models-pytorch 0.2.1 requires efficientnet-pytorch==0.6.3, but you have efficientnet-pytorch 0.7.1 which is incompatible.
/content/jpegio
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
/content/DocTamper/models
jpegio.read available: True
setup done. If jpegio.read is False, do Runtime > Restart and re-run this cell once.


In [2]:
# --- Stage data + pickles + backbones into the models/ working dir ---
# TamperDataset opens `lmdb.open(roots)` and reads `qt_table.pk` and `pks/{roots}_{minq}.pk`
# RELATIVE to the current dir, so everything must live in /content/DocTamper/models.
import os, glob, getpass
%cd /content/DocTamper/models

ROOTS = 'DocTamperV1-FCD'   # lmdb folder name == key used by pks/{ROOTS}_{minq}.pk
MINQ  = 75                  # must match an existing pks/{ROOTS}_{MINQ}.pk file

# 1) qt_table.pk and pks/ ship in the repo root.
if not os.path.exists('qt_table.pk'):
    !cp ../qt_table.pk ./
if not os.path.exists('pks'):
    !cp -r ../pks ./

# 2) LMDB dataset: unzip your archive here (password typed at runtime, not stored).
ZIP_PATH = '/content/drive/MyDrive/DocTamperV1-FCD.zip'
if not os.path.exists(f'{ROOTS}/data.mdb'):
    found = glob.glob(f'/content/**/{ROOTS}/data.mdb', recursive=True)
    if found:
        !ln -s "{os.path.dirname(found[0])}" "./{ROOTS}"
    else:
        assert os.path.exists(ZIP_PATH), f'Dataset zip not found: {ZIP_PATH}'
        pw = getpass.getpass('DocTamper zip password: ')
        !unzip -P "$pw" "{ZIP_PATH}" -d ./
        del pw
assert os.path.exists(f'{ROOTS}/data.mdb'), f'Expected {ROOTS}/data.mdb in models/ dir.'

# 3) Checkpoints (seg_dtd loads the backbones at construction). Download the author's folder:
#    https://drive.google.com/drive/folders/11Ep8PJIrlIveudQaRulDOBENHGqw762a
#    into MyDrive/checkpoints -> vph_imagenet.pt, swin_imagenet.pt, dtd_doctamper.pth
CKPT_DIR = '/content/drive/MyDrive/checkpoints'
for f in ['vph_imagenet.pt', 'swin_imagenet.pt', 'dtd_doctamper.pth']:
    if not os.path.exists(f) and os.path.exists(os.path.join(CKPT_DIR, f)):
        !cp "{CKPT_DIR}/{f}" ./
for f in ['vph_imagenet.pt', 'swin_imagenet.pt']:
    assert os.path.exists(f), (
        f'{f} not found. Download the checkpoints folder (link above) into {CKPT_DIR} -- '
        'seg_dtd loads these backbones when the model is built.')
print('staging done. backbones present; pks sample:', sorted(os.listdir('pks'))[:5], '...')

/content/DocTamper/models
staging done. backbones present; pks sample: ['DocTamperV1-FCD_75.pk', 'DocTamperV1-FCD_80.pk', 'DocTamperV1-FCD_85.pk', 'DocTamperV1-FCD_90.pk', 'DocTamperV1-SCD_75.pk'] ...


In [3]:
# --- Load TamperDataset from eval_dtd.py WITHOUT running its argparse/eval code ---
# eval_dtd.py is a script (top-level argparse + eval loop), so a plain import runs it and
# crashes on Colab's argv. We exec ONLY its imports + the dataset/metric classes.
import ast, torch
from torch.utils.data import DataLoader
%cd /content/DocTamper/models

src = open('eval_dtd.py').read()
tree = ast.parse(src)
keep = [n for n in tree.body
        if isinstance(n, (ast.Import, ast.ImportFrom))
        or (isinstance(n, ast.ClassDef) and n.name in ('TamperDataset', 'IOUMetric'))]
ns = {}
exec(compile(ast.Module(body=keep, type_ignores=[]), 'eval_dtd_extract', 'exec'), ns)
TamperDataset = ns['TamperDataset']
print('TamperDataset loaded (script body not executed)')

train_set = TamperDataset(ROOTS, mode='train', minq=MINQ)
# num_workers=0 surfaces errors directly (worker errors get wrapped); raise it once it runs.
train_loader = DataLoader(train_set, batch_size=4, shuffle=True, num_workers=0, drop_last=True)

batch = next(iter(train_loader))
print('batch keys:', list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f'  {k:6s} {tuple(v.shape)} {v.dtype}')
# Confirmed: image (B,3,512,512) f32 | label (B,1,512,512) i64 | rgb=DCT (B,512,512) i32
#            q (B,8,8) i64 quant table | i (B,) i64 quality

/content/DocTamper/models


/content/DocTamper/models/dtd.py:320: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast()


TamperDataset loaded (script body not executed)
batch keys: ['image', 'label', 'rgb', 'q', 'i']
  image  (4, 3, 512, 512) torch.float32
  label  (4, 1, 512, 512) torch.int64
  rgb    (4, 512, 512) torch.int32
  q      (4, 8, 8) torch.int64
  i      (4,) torch.int64


In [4]:
# --- Model + our combined loss + optimizer ---
import torch
from torch.cuda.amp import autocast, GradScaler

# Modern torch (>=2.6) defaults to torch.load(weights_only=True), but the author's backbones
# and checkpoint are FULL pickled objects -> force weights_only=False so the internal loads in
# dtd.py succeed. UNTRUSTED load: acceptable ONLY here (disposable session, author's files).
_ORIG_LOAD = torch.load
torch.load = lambda *a, **k: _ORIG_LOAD(*a, **{**k, 'weights_only': False})

# Use `import *` (like eval_dtd.py does) so the backbone classes (BasicLayer, etc.) live in
# __main__ -- the unpickler of vph_imagenet.pt / swin_imagenet.pt looks them up there.
from dtd import *
from src.losses import CombinedTamperLoss

device = 'cuda'
model = seg_dtd('', 2).to(device)      # loads vph_imagenet.pt / swin_imagenet.pt from cwd

# Optional: fine-tune from the released checkpoint instead of training from scratch.
FINETUNE_PTH = 'dtd_doctamper.pth'     # staged into models/ by the previous cell; None to skip
if FINETUNE_PTH and os.path.exists(FINETUNE_PTH):
    sd = torch.load(FINETUNE_PTH, map_location='cpu')['state_dict']
    sd = {k.replace('module.', ''): v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print('loaded checkpoint | missing:', len(missing), 'unexpected:', len(unexpected))

criterion = CombinedTamperLoss(lambda_dice=1.0, lambda_bound=0.5)  # ignore_index=100 (harmless on {0,1})
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scaler = GradScaler()
print('model + loss ready')

loaded checkpoint | missing: 0 unexpected: 0
model + loss ready


/tmp/ipykernel_22896/3797098987.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [6]:
import torch.nn as nn
n = 0
for m in model.modules():
    if isinstance(m, nn.GELU) and not hasattr(m, 'approximate'):
        m.approximate = 'none'; n += 1
print(f'patched {n} GELU modules')


patched 28 GELU modules


In [7]:
# --- Training loop (loss ablation: our CombinedTamperLoss) ---
import torch.nn.functional as F

def forward_dtd(model, batch):
    """Exactly mirrors eval_dtd.py's inference:
        data, dct_coef, qs = batch['image'], batch['rgb'], batch['q']
        pred = model(data, dct_coef, qs.unsqueeze(1))
    DCT stays int32 (embedding indices) and q is unsqueezed to (B,1,8,8). No float casts."""
    data = batch['image'].to(device)
    dct  = batch['rgb'].to(device)
    qs   = batch['q'].unsqueeze(1).to(device)
    return model(data, dct, qs)         # logits (B, 2, H, W)

EPOCHS, MAX_STEPS = 1, 200              # keep small for a first validation run
model.train()
for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader):
        if step >= MAX_STEPS:
            break
        target = batch['label'].squeeze(1).long().to(device)   # (B, H, W)
        optimizer.zero_grad()
        with autocast():
            logits = forward_dtd(model, batch)
            if logits.shape[-2:] != target.shape[-2:]:           # align if head outputs H/2
                logits = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False)
            loss, parts = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if step % 20 == 0:
            print(f'e{epoch} s{step:04d} | total {parts["total"]:.4f} '
                  f'ce {parts["ce"]:.4f} dice {parts["dice"]:.4f} bound {parts["boundary"]:.4f}')
print('training step finished')

/tmp/ipykernel_22896/2928768692.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


e0 s0000 | total 1.2788 ce 0.1163 dice 0.8025 bound 0.7202
e0 s0020 | total 0.9442 ce 0.0737 dice 0.5515 bound 0.6381
e0 s0040 | total 0.9178 ce 0.0556 dice 0.5212 bound 0.6821
e0 s0060 | total 0.6192 ce 0.0303 dice 0.3542 bound 0.4694
e0 s0080 | total 0.4952 ce 0.0284 dice 0.2310 bound 0.4714
e0 s0100 | total 0.5878 ce 0.0177 dice 0.3060 bound 0.5281
e0 s0120 | total 0.3908 ce 0.0112 dice 0.1316 bound 0.4958
e0 s0140 | total 0.4381 ce 0.0196 dice 0.1192 bound 0.5985
e0 s0160 | total 0.6346 ce 0.0617 dice 0.3086 bound 0.5284
e0 s0180 | total 0.3306 ce 0.0121 dice 0.1003 bound 0.4366
training step finished


## Phase 1b — add the OCR text prior (`TextPriorFusion`), no `dtd.py` edit

We fuse `M_text` into `decoder_output` (the feature feeding the segmentation head) by **wrapping the
segmentation head**, so the author's `dtd.py` stays untouched. Run the three cells below *after* the
baseline loop.

For a clean ablation, **re-run the model cell (cell 4) first** so this arm starts from the same
checkpoint as the baseline, rather than continuing the already-trained baseline weights.

OCR runs per batch here (slow); for full training, cache `M_text` per LMDB index once.

In [12]:
# === Phase 1b: wire the OCR text prior into the model (no dtd.py edit) ===
# Fuses M_text into `decoder_output` right before the segmentation head, by wrapping the head.
import easyocr
import torch.nn as nn
from src.text_prior import OCRTextMasker
from src.fusion import TextPriorFusion

reader = easyocr.Reader(['en'], gpu=True)
masker = OCRTextMasker(
    lambda im: [p for (p, _, _) in reader.readtext(im, detail=1, paragraph=False)],
    polygon=True, dilate=5)

class HeadWithPrior(nn.Module):
    """Fuse the current batch's text prior into the final decoder feature, then run the head.
    `text_mask` is set each step by the loop. Do NOT store a reference to the parent module here:
    assigning an nn.Module attribute registers it as a child, creating a .train() recursion cycle."""
    def __init__(self, head, in_ch):
        super().__init__()
        self.fusion = TextPriorFusion(in_ch)
        self.head = head
        self.text_mask = None      # plain Tensor attribute, NOT a submodule
    def forward(self, feat):
        return self.head(self.fusion(feat, self.text_mask))

dtd_core = model.model
# Unwrap any previous wrapper back to the original smp SegmentationHead, then (re)wrap.
head = dtd_core.segmentation_head
while hasattr(head, 'head'):
    head = head.head
C = head[0].in_channels            # smp SegmentationHead is a Sequential; [0] is the Conv2d
dtd_core.segmentation_head = HeadWithPrior(head, C).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)  # include fusion params
print('OCR-prior head wired; fusion in_channels =', C)

OCR-prior head wired; fusion in_channels = 64


In [13]:
# Build M_text per batch. The dataloader yields NORMALIZED image tensors, so de-normalize
# back to uint8 RGB for EasyOCR. (For real training, cache masks per LMDB index instead.)
import numpy as np
MEAN = np.array([0.485, 0.455, 0.406]); STD = np.array([0.229, 0.224, 0.225])

def batch_text_masks(images):                 # (B,3,H,W) normalized -> (B,1,H,W) float on device
    out = []
    for img in images:
        a = img.permute(1, 2, 0).cpu().numpy() * STD + MEAN
        out.append(masker(np.clip(a * 255, 0, 255).astype(np.uint8)))
    return torch.from_numpy(np.stack(out)[:, None]).float().to(device)

print('batch_text_masks ready')

batch_text_masks ready


In [14]:
# === Phase 1b loop: train WITH the OCR text prior (the +OCR ablation arm) ===
model.train()
for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader):
        if step >= MAX_STEPS:
            break
        dtd_core.segmentation_head.text_mask = batch_text_masks(batch['image'])   # set the prior
        target = batch['label'].squeeze(1).long().to(device)
        optimizer.zero_grad()
        with autocast():
            logits = forward_dtd(model, batch)
            if logits.shape[-2:] != target.shape[-2:]:
                logits = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False)
            loss, parts = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if step % 20 == 0:
            print(f'[+OCR] e{epoch} s{step:04d} | total {parts["total"]:.4f} '
                  f'ce {parts["ce"]:.4f} dice {parts["dice"]:.4f} bound {parts["boundary"]:.4f}')
print('OCR-prior training step finished')

/tmp/ipykernel_22896/908380790.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[+OCR] e0 s0000 | total 1.3405 ce 0.1966 dice 0.6502 bound 0.9874
[+OCR] e0 s0020 | total 0.4842 ce 0.0428 dice 0.1831 bound 0.5165
[+OCR] e0 s0040 | total 0.5051 ce 0.1028 dice 0.1826 bound 0.4394
[+OCR] e0 s0060 | total 0.4949 ce 0.0074 dice 0.2575 bound 0.4599
[+OCR] e0 s0080 | total 0.2409 ce 0.0170 dice 0.0392 bound 0.3693
[+OCR] e0 s0100 | total 0.2985 ce 0.0125 dice 0.1011 bound 0.3699
[+OCR] e0 s0120 | total 0.2264 ce 0.0104 dice 0.0331 bound 0.3660
[+OCR] e0 s0140 | total 0.7081 ce 0.1122 dice 0.2720 bound 0.6478
[+OCR] e0 s0160 | total 0.1459 ce 0.0037 dice 0.0243 bound 0.2358
[+OCR] e0 s0180 | total 0.1811 ce 0.0033 dice 0.0315 bound 0.2928
OCR-prior training step finished


## Notes & next steps

- **No `dtd.py` edit needed.** Phase 1b wraps the segmentation head, fusing the prior into the final
  decoder feature (`decoder_output`). A *deeper* injection (into an intermediate `MID` decoder feature
  `decoder_features["x_i_j"]`) would instead require editing `dtd.py`'s `MID.forward` — optional future work.
- **Ablations to report:** baseline (CE only) vs `CombinedTamperLoss` vs `+OCR prior`; plus a loss-term
  ablation (set `lambda_dice` / `lambda_bound` to 0). Run each from the same checkpoint (re-run cell 4),
  then evaluate with the DocTamper metrics (Pixel-F1, IoU, AUC).
- **Phase-2 (TA's note):** replace the external EasyOCR mask with a lightweight OCR/text head on the
  Visual Perception Head features (one backbone, two heads); select it by recall with `OCRCoverageMeter`.
- **Reminder:** never commit the dataset, `*.pth`, or `*.pk` — `.gitignore` blocks them; keep them in the
  Colab session / Drive only.